# Квантильное распределение поверх текущего ансамбля
Оценка использует сохранённые прогнозы текущего ансамбля на тех же временных окнах. Веса подбираются на трёх предтестовых неделях. Сравниваем смешивание с квантильной моделью и перенос только эффекта квантильной нормировки.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from catboost import CatBoostRegressor
from IPython.display import display

ROOT = Path(os.environ.get('BLEND_PACKAGE', '/root/experiments/session6_blend_server'))
OOF_PATH = None
TEST_PATH = None
OUT = ROOT / 'output_quantile_on_composite'
OUT.mkdir(parents=True, exist_ok=True)

BASES = ['session6', 'rf', 'cat_rich', 'xgb_rich', 'xgb_clean', 'profile28']
GROUP_KEYS = ['cutoff', 'horizon']
KEYS = GROUP_KEYS + ['route_id']
COLUMNS = KEYS + BASES + ['y', 'level24', 'current', 'delta1h', 'delta2h', 'network_current']
THREADS = 2
SEED = 42

def read_panel(path, name, start, end):
    archived = ROOT / 'blend_inputs' / 'blend_best_output' / name
    if path is None and archived.is_file():
        path = archived
    if path is None:
        candidates = [p for p in ROOT.rglob(name) if OUT not in p.parents
                      and set(COLUMNS).issubset(pq.ParquetFile(p).schema.names)]
        if len(candidates) != 1:
            raise ValueError(f'Укажите путь явно для {name}: {candidates}')
        path = candidates[0]
    frame = pd.read_parquet(path, columns=COLUMNS)
    frame['cutoff'] = pd.to_datetime(frame.cutoff)
    expected = pd.date_range(start, end, inclusive='left') + pd.Timedelta(hours=10, minutes=30)
    actual = pd.DatetimeIndex(frame.cutoff.unique()).sort_values()
    counts = frame.groupby(GROUP_KEYS).size()
    if (not actual.equals(expected) or frame.duplicated(KEYS).any()
            or set(frame.horizon.unique()) != set(range(1, 9))
            or len(counts) != len(expected)*8 or not counts.eq(1000).all()
            or not np.isfinite(frame[COLUMNS[3:]].to_numpy()).all()
            or frame.y.lt(0).any() or frame.session6.lt(0).any()):
        raise ValueError(f'Некорректная панель: {path}')
    print(path, len(frame))
    return frame.reset_index(drop=True)

DATA = ROOT/'blend_inputs/session6'
BASE_RUN = Path(os.environ.get('BLEND_BASE_RUN',str(ROOT/'output/blend_submit_948a52023607')))
PARENT = Path(os.environ.get('OVERLAP_H1_PARENT',str(ROOT/'output_overlap_composite/submission_overlap_composite_full_history.csv')))
RUN_TEST = BASE_RUN/'train_val/test_predictions.parquet'
RUN_FUTURE = BASE_RUN/'full_history/future_predictions.parquet'
STEP = pd.Timedelta(minutes=30)
MIN_TEST_GAIN = .0004


In [ ]:
def with_volume(frame, values, anchor=None):
    result = np.maximum(np.asarray(values, dtype=np.float64), 0)
    reference = frame.session6.to_numpy(dtype=np.float64) if anchor is None else np.asarray(anchor, dtype=np.float64)
    totals = frame[GROUP_KEYS].assign(pred=result, reference=reference).groupby(GROUP_KEYS, sort=False).transform('sum')
    reference_total = totals.reference.to_numpy()
    prediction_total = totals.pred.to_numpy()
    out = np.divide(result * reference_total, prediction_total, out=reference.copy(), where=prediction_total > 0)
    predicted = frame[GROUP_KEYS].assign(value=out).groupby(GROUP_KEYS, sort=False).value.sum()
    expected = frame[GROUP_KEYS].assign(value=reference).groupby(GROUP_KEYS, sort=False).value.sum()
    if not np.isfinite(out).all() or not np.allclose(predicted, expected, rtol=1e-10, atol=1e-5):
        raise ValueError('Не удалось сохранить заданный объём')
    return out

def shape_matrix(frame):
    return np.column_stack([with_volume(frame, frame[name]) for name in BASES])

def fit_weights(x, y):
    y = np.asarray(y, dtype=np.float64)
    denominator = y.sum()
    if denominator <= 0:
        raise ValueError('Нулевая сумма фактов для подбора весов')
    weights = np.zeros(x.shape[1])
    weights[0] = 1
    prediction = x[:, 0].copy()
    best = np.abs(prediction - y).sum() / denominator
    for step in (.2, .05):
        for _ in range(20):
            choice = None
            for source in np.flatnonzero(weights >= step - 1e-10):
                for destination in range(len(weights)):
                    if source == destination:
                        continue
                    amount = min(step, weights[source])
                    trial = prediction + amount * (x[:, destination] - x[:, source])
                    loss = np.abs(trial - y).sum() / denominator
                    if loss < best - 1e-9:
                        best = loss
                        choice = source, destination, amount, trial
            if choice is None:
                break
            source, destination, amount, prediction = choice
            weights[source] -= amount
            weights[destination] += amount
    return weights

def meta_features(frame, matrix, baseline):
    level = np.maximum(frame.level24.to_numpy(dtype=np.float64), 1)
    context = pd.DataFrame(dict(
        log_level=np.log1p(level), current=frame.current.to_numpy() / level,
        delta1h=frame.delta1h.to_numpy() / level, delta2h=frame.delta2h.to_numpy() / level,
        log_network=np.log1p(frame.network_current.to_numpy()),
        horizon=frame.horizon.to_numpy(), dayofweek=frame.cutoff.dt.day_of_week.to_numpy(),
        shape_disagreement=matrix.std(axis=1) / level, blend=baseline / level))
    for index, name in enumerate(BASES):
        context[f'{name}_at_anchor_volume'] = matrix[:, index] / level
        context[f'{name}_minus_session6'] = (matrix[:, index] - matrix[:, 0]) / level
    return context.clip(-50, 50).to_numpy(dtype=np.float32), level

def metric(y, prediction):
    actual = np.asarray(y, dtype=np.float64).ravel()
    predicted = np.asarray(prediction, dtype=np.float64).ravel()
    volume = actual.sum()
    if actual.shape != predicted.shape or not np.isfinite(actual).all() or not np.isfinite(predicted).all():
        raise ValueError('Некорректный вход метрики')
    if volume == 0:
        return dict(rows=len(actual), WAPE=np.nan, relative_bias=np.nan, score=np.nan)
    error = predicted - actual
    wape = np.abs(error).sum() / volume
    bias = error.sum() / volume
    return dict(rows=len(actual), WAPE=wape, relative_bias=bias, score=wape + abs(bias))

def fit_meta(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix, frame.y)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    model = CatBoostRegressor(depth=4, iterations=220, loss_function='MAE', learning_rate=.035,
                              l2_leaf_reg=20, random_seed=SEED, verbose=False, allow_writing_files=False,
                              task_type='CPU', thread_count=THREADS)
    model.fit(features, (frame.y.to_numpy(dtype=float)-baseline)/level, sample_weight=level/level.mean())
    return model, weights

def predict_meta(frame, fitted):
    model, weights = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    features, level = meta_features(frame, matrix, baseline)
    return with_volume(frame, baseline + np.clip(model.predict(features), -.5, .5)*level)

def score_mix(frame, shape, weight):
    prediction = with_volume(frame, (1-weight)*frame.session6.to_numpy() + weight*shape)
    daily = pd.DataFrame([dict(day=str(day.date()), **metric(group.y, prediction[group.index]))
                          for day, group in frame.groupby('cutoff')])
    summary = dict(**metric(frame.y, prediction), daily_mean=daily.score.mean(),
                   daily_p90=daily.score.quantile(.9), daily_worst=daily.score.max())
    return summary, daily, prediction


In [ ]:
def load_target(path):
    raw = pd.read_parquet(path,columns=['timestamp','route_id','target_1h'])
    panel = raw.pivot(index='timestamp',columns='route_id',values='target_1h').sort_index().sort_index(axis=1)
    if (panel.shape[1]!=1000 or not panel.index.equals(pd.date_range(panel.index[0],panel.index[-1],freq=STEP))
            or not np.isfinite(panel.to_numpy()).all() or panel.lt(0).any().any()):
        raise ValueError('Некорректная история цели')
    return panel

def overlap_features(frame,level):
    x = np.zeros((len(frame),5),dtype=np.float32)
    mask = frame.horizon.eq(1).to_numpy()
    if not mask.any():
        return x
    selected = frame.loc[mask]
    positions = TARGET_PANEL.index.get_indexer(pd.DatetimeIndex(selected.cutoff))
    routes = TARGET_PANEL.columns.get_indexer(selected.route_id)
    if np.any(positions<7) or np.any(routes<0):
        raise ValueError('Не хватает причинной истории h1')
    seq = TARGET_PANEL.to_numpy(dtype=np.float64,copy=False)[positions[:,None]-np.arange(7,-1,-1),routes[:,None]]
    low = np.zeros(len(seq))
    high = seq[:,0].copy()
    valid = np.ones(len(seq),dtype=bool)
    for value in seq[:,1:].T:
        low,high = np.maximum(0,value-high),np.minimum(value,value-low)
        valid &= low<=high+1e-6
    low = np.where(valid,low,0.)
    high = np.where(valid,np.maximum(high,low),seq[:,-1])
    midpoint = (high+low)/2
    x[mask] = np.column_stack([low,high-low,midpoint,
                               midpoint-np.divide(seq[:,-1],2),valid*level[mask]])/level[mask,None]
    return x

def design(frame,matrix,baseline,augmented):
    context,level = meta_features(frame,matrix,baseline)
    if augmented:
        context = np.column_stack([context,overlap_features(frame,level)])
    return np.clip(context,-50,50).astype(np.float32,copy=False),level

def train_pair(frame):
    matrix = shape_matrix(frame)
    weights = fit_weights(matrix,frame.y)
    baseline = matrix @ weights
    models = {}
    for name,augmented in [('control',False),('overlap_h1',True)]:
        features,level = design(frame,matrix,baseline,augmented)
        model = CatBoostRegressor(depth=4,iterations=220,loss_function='MAE',learning_rate=.035,
                                  l2_leaf_reg=20,random_seed=SEED,verbose=False,allow_writing_files=False,
                                  task_type='CPU',thread_count=4)
        model.fit(features,(frame.y.to_numpy(dtype=float)-baseline)/level,sample_weight=level/level.mean())
        models[name] = model
    return weights,models

def predict_pair(frame,fitted):
    weights,models = fitted
    matrix = shape_matrix(frame)
    baseline = matrix @ weights
    result = {}
    for name,model in models.items():
        features,level = design(frame,matrix,baseline,name=='overlap_h1')
        residual = np.clip(model.predict(features),-.5,.5)*level
        result[name] = with_volume(frame,baseline+residual)
    if 'control' in result and 'overlap_h1' in result:
        result['overlap_h1'] = np.where(frame.horizon.eq(1),result['overlap_h1'],result['control'])
    return result

def summarize(frame,forecasts,fold,period):
    rows = []
    h1 = frame.horizon.eq(1).to_numpy()
    for name,prediction in forecasts.items():
        first = metric(frame.y.to_numpy()[h1],prediction[h1])
        daily = [metric(g.y,prediction[g.index])['score'] for _,g in frame.groupby('cutoff')]
        rows.append(dict(fold=fold,period=period,variant=name,**metric(frame.y,prediction),
                         h1_WAPE=first['WAPE'],h1_score=first['score'],
                         daily_mean=float(np.mean(daily)),daily_p90=float(np.quantile(daily,.9)),
                         daily_worst=float(np.max(daily))))
    return rows



RECIPES = ('qmatch','median_scale')
MIXES = (.35,.65,1.)
QUANTILES = np.array([.12,.26,.40,.50,.60,.74,.88])

def quantile_inputs(frame,matrix,baseline):
    context,level = meta_features(frame,matrix,baseline)
    dates = TARGET_PANEL.index.get_indexer(pd.DatetimeIndex(frame.cutoff))
    routes = TARGET_PANEL.columns.get_indexer(frame.route_id)
    horizons = frame.horizon.to_numpy(dtype=int)
    if (dates<672).any() or (routes<0).any():
        raise ValueError('Недостаточно причинной истории')
    values = TARGET_PANEL.to_numpy(copy=False)
    weekly = values[dates[:,None]+horizons[:,None]-48*np.array([1,2,3,7,14]),routes[:,None]]
    recent = values[dates[:,None]-np.array([0,1,2,4,6,8]),routes[:,None]]
    scale = np.column_stack([weekly.mean(axis=1),weekly.std(axis=1),recent.mean(axis=1),recent.std(axis=1)])/level[:,None]
    return np.column_stack([context,scale,overlap_features(frame,level)]).clip(-50,50).astype(np.float32),level

def fit_challengers(frame,names):
    control = train_pair(frame)
    if not names:
        return control,[]
    matrix = shape_matrix(frame)
    baseline = matrix @ control[0]
    features,level = quantile_inputs(frame,matrix,baseline)
    target = (frame.y.to_numpy()-baseline)/level
    models = []
    for alpha in QUANTILES:
        model = CatBoostRegressor(depth=5,iterations=320,task_type='GPU',devices='0',
                                  loss_function=f'Quantile:alpha={alpha:g}',learning_rate=.04,
                                  l2_leaf_reg=25,random_seed=SEED,thread_count=4,verbose=False,
                                  allow_writing_files=False)
        model.fit(features,target,sample_weight=level/level.mean())
        print('Квантиль',alpha,'строк',len(frame),flush=True)
        models.append(model)
    return control,models

def quantile_allocation(frame,values):
    result = np.zeros(len(frame))
    alpha_used = []
    anchor = frame.session6.to_numpy()
    for _,g in frame.groupby(GROUP_KEYS,sort=False):
        indices = g.index.to_numpy()
        grid = values[indices]
        target = anchor[indices].sum()
        total = grid.sum(axis=0)
        alpha = np.interp(target,total,QUANTILES)
        upper = int(np.clip(np.searchsorted(QUANTILES,alpha,side='right'),1,len(QUANTILES)-1))
        weight = (alpha-QUANTILES[upper-1])/(QUANTILES[upper]-QUANTILES[upper-1])
        result[indices] = (1-weight)*grid[:,upper-1]+weight*grid[:,upper]
        alpha_used.append(alpha)
    return with_volume(frame,result),np.array(alpha_used)

def predict_challengers(frame,fitted):
    control,models = fitted
    reference = predict_pair(frame,control)['overlap_h1']
    forecasts = {'overlap_h1':reference}
    if not models:
        return forecasts
    matrix = shape_matrix(frame)
    baseline = matrix @ control[0]
    features,level = quantile_inputs(frame,matrix,baseline)
    raw = np.sort(np.column_stack([np.maximum(baseline+model.predict(features)*level,0.) for model in models]),axis=1)
    adapted,alphas = quantile_allocation(frame,raw)
    scaled = with_volume(frame,raw[:,3])
    print('Медиана квантиля для распределения',float(np.median(alphas)),flush=True)
    for name,expert in [('qmatch',adapted),('median_scale',scaled)]:
        for weight in MIXES:
            forecasts[f'{name}@{weight}'] = reference+weight*(expert-reference)
    return forecasts

def compose(frame,fitted,prior):
    forecasts = predict_challengers(frame,fitted)
    reference = np.asarray(prior)
    if len(reference)!=len(frame) or not np.isfinite(reference).all():
        raise ValueError('Некорректный контроль')
    if 'qmatch@1.0' not in forecasts:
        return {'composite':reference}
    match = forecasts['qmatch@1.0']
    scale = forecasts['median_scale@1.0']
    output = {'composite':reference}
    for weight in (.25,.5,.75,1.):
        output[f'combine@{weight}'] = with_volume(frame,reference+weight*(match-reference),anchor=reference)
        output[f'normalize@{weight}'] = with_volume(frame,reference+weight*(match-scale),anchor=reference)
    return output

def load_composite(frame,boundary):
    path = ROOT/'output_overlap_composite/historical_predictions'/f'{boundary}.parquet'
    cached = pd.read_parquet(path,columns=KEYS+['y','composite']).rename(columns={'y':'cache_y'})
    joined = frame[KEYS+['y']].merge(cached,on=KEYS,validate='one_to_one',how='left',sort=False)
    if joined.composite.isna().any() or not np.array_equal(joined.y,joined.cache_y):
        raise ValueError('Не совпали ключи кэша ансамбля')
    return joined.composite.to_numpy()


In [ ]:

TARGET_PANEL = load_target(DATA/'train_solo_track.parquet')
oof = read_panel(OOF_PATH,'base_oof.parquet','2025-09-20','2025-10-18')
rows = []
for start,end in [('2025-09-27','2025-10-04'),('2025-10-04','2025-10-11'),('2025-10-11','2025-10-18')]:
    train = oof.loc[oof.cutoff+oof.horizon*STEP < pd.Timestamp(start)].reset_index(drop=True)
    val = oof.loc[oof.cutoff.between(start,end,inclusive='left')].reset_index(drop=True)
    print('Валидация',start,flush=True)
    fitted = fit_challengers(train,['quantiles'])
    rows.extend(summarize(train,predict_challengers(train,fitted),start,'train_oof'))
    rows.extend(summarize(val,compose(val,fitted,load_composite(val,start)),start,'val'))
validation = pd.DataFrame(rows)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
scores = validation.loc[validation.period.eq('val')].pivot(index='fold',columns='variant',values='daily_mean')
gains = -scores.sub(scores.composite,axis=0)
rank = gains.loc[:,gains.gt(0).all()].mean().sort_values(ascending=False)
chosen = rank.index[0] if len(rank) else None
(OUT/'frozen_recipe.json').write_text(json.dumps(dict(chosen=chosen,test_used_for_tuning=False,min_test_gain=MIN_TEST_GAIN),indent=2))
display(gains)
print('Зафиксированный кандидат',chosen,flush=True)


In [ ]:

test = read_panel(RUN_TEST,'test_predictions.parquet','2025-10-18','2025-11-01')
fitted = fit_challengers(oof,['quantiles'])
rows.extend(summarize(oof,predict_challengers(oof,fitted),'2025-10-18','train_oof'))
predictions = compose(test,fitted,load_composite(test,'2025-10-18'))
selected = {name:predictions[name] for name in ('composite',chosen) if name is not None}
rows.extend(summarize(test,selected,'2025-10-18','test'))
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT/'metrics.csv',index=False)
display(metrics.loc[metrics.fold.eq('2025-10-18')])
evaluation = metrics.loc[metrics.period.eq('test')].set_index('variant')
gain = float(evaluation.loc['composite','daily_mean']-evaluation.loc[chosen,'daily_mean']) if chosen else None
promoted = bool(chosen and gain>=MIN_TEST_GAIN and evaluation.loc[chosen,'score']<evaluation.loc['composite','score'])
(OUT/'decision.json').write_text(json.dumps(dict(chosen=chosen,promoted=promoted,test_daily_gain=gain,test_before_full_refit=True),indent=2))
print('Решение',promoted,'выигрыш',gain,flush=True)


In [ ]:

if promoted:
    future = pd.read_parquet(RUN_FUTURE,columns=[x for x in COLUMNS if x!='y'])
    future['cutoff'] = pd.to_datetime(future.cutoff)
    request = pd.read_parquet(DATA/'test_solo_track.parquet',columns=['id','route_id','timestamp'])
    parent = pd.read_csv(PARENT)
    cutoff = request.timestamp.min()-STEP
    history = pd.concat([oof,test],ignore_index=True)
    if (len(parent)!=8000 or len(request)!=8000 or len(future)!=8000 or not parent.id.is_unique
            or not request.id.is_unique or set(parent.id)!=set(request.id) or future.duplicated(KEYS).any()
            or not future.cutoff.eq(cutoff).all() or TARGET_PANEL.index[-1]!=cutoff
            or not (history.cutoff+history.horizon*STEP<=cutoff).all()):
        raise ValueError('Несовместимые данные для полного обучения')
    fitted_final = fit_challengers(history,['quantiles'])
    display(pd.DataFrame(summarize(history,predict_challengers(history,fitted_final),'full_history','train_oof')))
    keys = request.merge(parent,on='id',validate='one_to_one',sort=False)
    keys['cutoff'] = cutoff
    keys['horizon'] = ((keys.timestamp-cutoff)/STEP).astype(int)
    parent_aligned = future[KEYS].merge(keys[KEYS+['y_pred']],on=KEYS,validate='one_to_one',how='left',sort=False)
    future['candidate'] = compose(future,fitted_final,parent_aligned.y_pred.to_numpy())[chosen]
    query = keys.merge(future[KEYS+['candidate']],on=KEYS,validate='one_to_one',how='left',sort=False)
    if query.candidate.isna().any() or not query.id.equals(request.id):
        raise ValueError('Ключи конкурса не совпали')
    destination = OUT/'submission_quantile_on_composite_full_history.csv'
    query[['id']].assign(y_pred=query.candidate).to_csv(destination.with_suffix('.partial'),index=False)
    destination.with_suffix('.partial').replace(destination)
    for alpha,model in zip(QUANTILES,fitted_final[1]):
        model.save_model(str(OUT/f'quantile_{alpha:g}.cbm'))
    print('Сабмит',destination,flush=True)
print('Результаты',OUT,flush=True)
